In [ ]:
# ── Environment Setup ─────────────────────────────────────────────────────
import os
import sys
from pathlib import Path

# Set this to False if you intentionally want to run locally.
KAGGLE_ONLY = True
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if KAGGLE_ONLY and not IS_KAGGLE:
    raise RuntimeError(
        "This notebook is configured to run on Kaggle only.\n"
        "Open it on Kaggle, or set KAGGLE_ONLY=False in the first cell."
    )

WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()
os.chdir(WORK_DIR)
sys.path.append(str(WORK_DIR))

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# Install dependencies (Kaggle)
if IS_KAGGLE:
    os.system("pip install transformers datasets sacrebleu sentencepiece -q")

import torch
if torch.cuda.is_available():
    DEVICE, USE_FP16 = "cuda", True
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE, USE_FP16 = "mps", False
else:
    DEVICE, USE_FP16 = "cpu", False

print("Environment : Kaggle" if IS_KAGGLE else "Environment : Local")
print(f"DEVICE      : {DEVICE}  |  USE_FP16: {USE_FP16}")
print(f"ROOT        : {ROOT}")

# Experiment 1 — Encoder–Decoder (MarianMT)

**Goal:**
- Run the pretrained model `Helsinki-NLP/opus-mt-en-vi` (no fine-tuning) to get a **baseline BLEU**
- Fine-tune on cleaned IWSLT15 EN–VI and report **BLEU after fine-tuning**
- Compare results and save artifacts to `results/`

**Architecture:** MarianMT is an Encoder–Decoder Transformer (Vaswani et al., 2017), pretrained on the OPUS corpus.

In [ ]:
# Check GPU
if DEVICE == "cuda":
    !nvidia-smi
else:
    print(f"No NVIDIA GPU — running on: {DEVICE}")

## Block 1: Imports

We use Hugging Face Transformers (industry-standard NLP tooling) and `sacrebleu` to compute BLEU.

In [ ]:
import os
import json
import numpy as np
import torch
import sacrebleu
from pathlib import Path
from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
 )

os.makedirs("results", exist_ok=True)
print("PyTorch version:", torch.__version__)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## Block 2: Load Data (from Day 1)

The dataset was cleaned and saved in Hugging Face Arrow format. Columns: `en` (English) and `vi` (Vietnamese).

In [ ]:
from pathlib import Path
import shutil
import os

# Automatically find the data directory
if IS_KAGGLE:
    print("Searching for data in Kaggle Input...")
    # Find all directories named 'iwslt15_cleaned' inside /kaggle/input
    found_paths = list(Path("/kaggle/input").rglob("iwslt15_cleaned"))
    
    if not found_paths:
        raise FileNotFoundError(
            "Could not find the 'iwslt15_cleaned' directory! "
            "Make sure you have clicked 'Add Data' and added the output of Notebook 01."
        )
    
    source_path = str(found_paths[0])
    dataset_path = "/kaggle/working/iwslt15_cleaned"
    
    # Copy to a writable directory (/kaggle/working)
    if not os.path.exists(dataset_path):
        print(f"Copying data from {source_path} to {dataset_path} to avoid Read-Only error...")
        shutil.copytree(source_path, dataset_path)
    else:
        print(f"Data already copied to {dataset_path}")

else:
    dataset_path = "data/processed/iwslt15_cleaned"

# Load data
dataset = load_from_disk(dataset_path)

print(dataset)
print("\n--- Inspect 1 sample ---")
print("Fields:", dataset["train"].column_names)
sample = dataset["train"][0]
print("EN:", sample["en"])
print("VI:", sample["vi"])

In [ ]:
# Dataset sizes
print(f"Train: {len(dataset['train']):,} samples")
print(f"Validation: {len(dataset['validation']):,} samples")
print(f"Test: {len(dataset['test']):,} samples")

## Block 3: Load Model + Tokenizer

MarianMT (`Helsinki-NLP/opus-mt-en-vi`) is an encoder–decoder Transformer pretrained on OPUS for English–Vietnamese.

In [ ]:
model_name = "Helsinki-NLP/opus-mt-en-vi"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Parameter count
total_params = sum(p.numel() for p in model.parameters())
print(f"Model: {model_name}")
print(f"Total parameters: {total_params:,}")

## Block 4: Baseline BLEU (No Fine-tuning)

Run the pretrained model on the test set to get a baseline BLEU score — this is the reference point for comparison after fine-tuning.

In [ ]:
# Prepare test split — the dataset has flat 'en' and 'vi' columns
source_texts = dataset["test"]["en"]
references = dataset["test"]["vi"]

print(f"Test samples: {len(source_texts)}")
print(f"Example source: {source_texts[0]}")
print(f"Example reference: {references[0]}")

In [ ]:
# Translate the test set using model.generate() directly
# (pipeline('translation') was removed/changed in newer transformers versions)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

baseline_predictions = []
batch_size = 32

with torch.no_grad():
    for i in range(0, len(source_texts), batch_size):
        batch = source_texts[i:i + batch_size]
        inputs = tokenizer(
            batch, return_tensors='pt', padding=True,
            truncation=True, max_length=128
        ).to(device)
        outputs = model.generate(**inputs, max_length=128)
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        baseline_predictions.extend(decoded)
        if (i // batch_size) % 5 == 0:
            print(f'  Processed {min(i + batch_size, len(source_texts))}/{len(source_texts)}')

baseline_bleu = sacrebleu.corpus_bleu(baseline_predictions, [references])
print(f'\nBaseline BLEU: {baseline_bleu.score:.2f}')


In [ ]:
# Save baseline predictions (intermediate checkpoint)
with open("results/baseline_predictions.json", "w", encoding="utf-8") as f:
    json.dump({
        "bleu": baseline_bleu.score,
        "predictions": baseline_predictions[:50]  # Save 50 samples for quick debugging
    }, f, ensure_ascii=False, indent=2)
print("Saved baseline_predictions.json")

## Block 5: Tokenize Data for Fine-tuning

`max_length=128` covers ~95% of IWSLT15 sentence lengths (based on Day 1 EDA). The dataset uses flat `en`/`vi` columns (not a nested `translation` dict).

In [ ]:
# Reload the model for fine-tuning (reset to pretrained weights)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
torch.cuda.empty_cache()


In [ ]:
max_length = 128

def preprocess_function(examples):
    return tokenizer(
        text=examples["en"],
        text_target=examples["vi"],
        max_length=max_length,
        truncation=True,
        padding="max_length",
    )

tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing",
 )
print("Tokenized dataset:", tokenized_dataset)

## Block 6: Training Configuration

Key hyperparameters:
- `learning_rate=2e-5` — standard choice for fine-tuning Transformers
- `num_train_epochs=3` — usually enough to converge without heavy overfitting
- `per_device_train_batch_size=16` — reduce to 8 if you hit OOM

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./results-encoder-decoder",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy="epoch",
    predict_with_generate=True,
    fp16=USE_FP16,
    logging_steps=100,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
 )
print("Training arguments configured.")

## Block 7: BLEU Metric Function

We compute BLEU using `sacrebleu.corpus_bleu`, a standard MT evaluation metric (WMT-style).

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # Replace -100 (ignored label positions) with pad_token_id before decoding
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = sacrebleu.corpus_bleu(decoded_preds, [decoded_labels])
    return {"bleu": result.score}

## Block 8: Train

Estimated time: **~1–2 hours** on Colab T4 for ~130k pairs (3 epochs).

If Colab disconnects, you can resume from a checkpoint:
- `trainer.train(resume_from_checkpoint="./results-encoder-decoder/checkpoint-xxx")`

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
# Save training logs (for analysis/reporting)
train_log = trainer.state.log_history
with open("results/training_log_exp1.json", "w") as f:
    json.dump(train_log, f, indent=2)
print("Saved training_log_exp1.json")

## Block 9: Evaluate After Fine-tuning

Translate the test set again using the fine-tuned model and compare BLEU against the baseline.

In [ ]:
# Move the model to GPU (if available) and set eval mode
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()
model.to(device)

finetuned_predictions = []
batch_size = 32

with torch.no_grad():
    for i in range(0, len(source_texts), batch_size):
        batch = source_texts[i:i + batch_size]
        inputs = tokenizer(
            batch, return_tensors="pt", padding=True,
            truncation=True, max_length=128,
        ).to(device)
        outputs = model.generate(**inputs, max_length=128, num_beams=4)
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        finetuned_predictions.extend(decoded)
        if (i // batch_size) % 5 == 0:
            print(f"  Processed {min(i + batch_size, len(source_texts))}/{len(source_texts)}")

finetuned_bleu = sacrebleu.corpus_bleu(finetuned_predictions, [references])
print(f"\nBaseline BLEU:   {baseline_bleu.score:.2f}")
print(f"Fine-tuned BLEU: {finetuned_bleu.score:.2f}")
print(f"Improvement:     +{finetuned_bleu.score - baseline_bleu.score:.2f}")

## Block 10: Save Results

Save metrics to `results/exp1_encoder_decoder.json` and a few translation examples to `results/translation_examples.json`.

In [ ]:
# Save metrics/results
results = {
    "model": model_name,
    "baseline_bleu": baseline_bleu.score,
    "finetuned_bleu": finetuned_bleu.score,
    "improvement": finetuned_bleu.score - baseline_bleu.score,
    "hyperparameters": {
        "learning_rate": 2e-5,
        "epochs": 3,
        "batch_size": 16,
        "max_length": 128,
        "weight_decay": 0.01,
        "fp16": True,
    },
    "dataset": "iwslt15_cleaned",
    "test_samples": len(source_texts),
}

with open("results/exp1_encoder_decoder.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved exp1_encoder_decoder.json")
print(json.dumps(results, indent=2))

In [ ]:
# Save translation examples for qualitative analysis (Day 3)
examples = []
for i in range(20):
    examples.append({
        "source": source_texts[i],
        "reference": references[i],
        "baseline": baseline_predictions[i],
        "finetuned": finetuned_predictions[i],
    })

with open("results/translation_examples.json", "w", encoding="utf-8") as f:
    json.dump(examples, f, ensure_ascii=False, indent=2)
print("Saved translation_examples.json")

# Print a few examples for a quick sanity check
for ex in examples[:3]:
    print("\nSRC:", ex["source"])
    print("REF:", ex["reference"])
    print("BASE:", ex["baseline"])
    print("FINE:", ex["finetuned"])

In [ ]:
# Summary
print("=" * 50)
print("EXPERIMENT 1 COMPLETED")
print("=" * 50)
print(f"Baseline BLEU:   {baseline_bleu.score:.2f}")
print(f"Fine-tuned BLEU: {finetuned_bleu.score:.2f}")
print(f"Improvement:     +{finetuned_bleu.score - baseline_bleu.score:.2f}")
print("\nFiles saved:")
print("  results/exp1_encoder_decoder.json")
print("  results/translation_examples.json")
print("  results/baseline_predictions.json")
print("  results/training_log_exp1.json")